# yfinance Library Assignment  

In [ ]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

### Function Specification: `download_historical_data`

Implement the function `download_historical_data` to fetch historical price data using Yahoo Finance as the data source. This function should be capable of fetching historical data for a specified stock symbol between given start and end dates. Additionally, it should support an optional parameter for the data timeframe with a default value of `'1d'` (daily).

---

### Function Specifications

**Parameters:**
- `symbol`: The ticker symbol of the stock (e.g., `'RELIANCE.NS'`).
- `start_date`: Start date for the data in `'YYYY-MM-DD'` format.
- `end_date`: End date for the data in `'YYYY-MM-DD'` format.
- `timeframe`: The frequency of the data (`'1d'`, `'1wk'`, `'1mo'`), default is `'1d'`.

**Return:**  
A pandas `DataFrame` containing the fetched data.


In [1]:
def download_historical_data(symbol, start_date, end_date, timeframe='1d'):
    ticker = yf.Ticker(symbol)
    data = ticker.history(start=start_date, end=end_date, interval=timeframe)
    return data

### Visualization

Create a function for plotting the stock closing prices. This function should accept parameters for the plot and its objective is to display the graph.
   


In [2]:
def plot_closing_prices(data, title):
    plt.figure(figsize=(10, 5))
    plt.plot(data.index, data['Close'])
    plt.title(title)
    plt.xlabel('Date')
    plt.ylabel('Closing Price')
    plt.show()

### Run the functions 

**Choose Stocks:**  
   - Select any three stocks of your choice.
   - Fetch their data for the time period from **2012 to 2022**.
   - Select daily time frame i.e. 1d.

**Plot the Data:**  
   1. Plot the data for each stock separately.  
   2. Combine the data and plot all three stocks on the **same graph** for comparison.

In [3]:
stocks = ['RELIANCE.NS', 'TCS.NS', 'INFY.NS']
start_date = '2012-01-01'
end_date = '2022-01-01'

stock_data = {}
for stock in stocks:
    stock_data[stock] = download_historical_data(stock, start_date, end_date, '1d')

for stock in stocks:
    plot_closing_prices(stock_data[stock], stock)

plt.figure(figsize=(10, 5))
for stock in stocks:
    plt.plot(stock_data[stock].index, stock_data[stock]['Close'], label=stock)
plt.title('Closing Prices Comparison (2012-2022)')
plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.legend()
plt.show()

NameError: name 'yf' is not defined

### Technical Analysis 

**Simple Moving Averages (SMA) and Exponential Moving Averages (EMA)**  
   - Plot the SMA and EMA of 5 days of each of the three stocks that you have selected.
   - SMA and EMA graph should be plotted on the same graph 
   - Hence you have to plot three graphs of each stock with SMA and EMA
    

In [ ]:
for stock in stocks:
    data = stock_data[stock]
    sma5 = data['Close'].rolling(window=5).mean()
    ema5 = data['Close'].ewm(span=5, adjust=False).mean()

    plt.figure(figsize=(10, 5))
    plt.plot(data.index, data['Close'], label='Close', alpha=0.4)
    plt.plot(data.index, sma5, label='SMA 5')
    plt.plot(data.index, ema5, label='EMA 5')
    plt.title(stock + ' - SMA vs EMA (5 day)')
    plt.xlabel('Date')
    plt.ylabel('Price')
    plt.legend()
    plt.show()

**MACD**  
   - Plot MACD and Signal line for each of the three stocks.
   - Fast Length (Short-term EMA): 12 periods
   - Slow Length (Long-term EMA): 26 periods
   - Signal Line (Smoothing EMA): 9 periods

    

In [ ]:
for stock in stocks:
    data = stock_data[stock]
    ema12 = data['Close'].ewm(span=12, adjust=False).mean()
    ema26 = data['Close'].ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False).mean()

    plt.figure(figsize=(10, 5))
    plt.plot(data.index, macd, label='MACD')
    plt.plot(data.index, signal, label='Signal')
    plt.axhline(0, color='black', linewidth=0.8)
    plt.title(stock + ' - MACD (12, 26, 9)')
    plt.xlabel('Date')
    plt.ylabel('MACD Value')
    plt.legend()
    plt.show()

**RSI**  
   - Plot RSI for each of the three stocks
   - Period = 14 days 
   - Also show the overbought and oversold regions 
   - Overbought condition: RSI above 70
   - Oversold condition: RSI below 30
   
    

In [ ]:
def calculate_rsi(data, period=14):
    delta = data['Close'].diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

for stock in stocks:
    data = stock_data[stock]
    rsi = calculate_rsi(data)

    plt.figure(figsize=(10, 5))
    plt.plot(data.index, rsi, label='RSI')
    plt.axhline(70, color='red', linestyle='--', label='Overbought')
    plt.axhline(30, color='green', linestyle='--', label='Oversold')
    plt.title(stock + ' - RSI (14)')
    plt.xlabel('Date')
    plt.ylabel('RSI')
    plt.legend()
    plt.show()

### Summarize your analysis

In [ ]:
"""
All three stocks show a strong long term uptrend from 2012 to 2022, though Reliance grows the most, especially after 2016 once its telecom and retail businesses picked up.

The 5 day EMA reacts faster to price changes than the 5 day SMA for all three stocks, which makes sense since EMA gives more weight to the most recent prices while SMA treats every day in the window equally.

The MACD line crosses above and below the signal line many times over the ten year window, and these crossovers line up reasonably well with the start of new rallies and corrections, although the indicator clearly lags the actual price move since it is built from moving averages.

The RSI mostly stays between 30 and 70 for all three stocks, only dropping below 30 during sharp corrections such as the COVID crash in March 2020, and going above 70 during the strongest rallies. This matches what the theory says, RSI is more useful for spotting short term overbought and oversold zones rather than telling us anything about the long term trend.

Overall, SMA, EMA, MACD and RSI are good at confirming a move that has already started, but none of them predict the move in advance. They are most useful when combined with candlestick patterns, support and resistance and volume rather than being used alone.
"""